# Advanced DOT Traffic Analysis 
This pipeline ingests NYC Yellow Taxi data, maps it to real neighborhoods, categorizes trips into Rush Hour windows, and joins an external Weather API to prove how rain impacts congestion.

In [5]:
import pandas as pd
import numpy as np
import requests

# --- 1. INGEST MULTIPLE SOURCES ---
print("Loading Parquet and CSV data...")
df_trips = pd.read_parquet('data/yellow_tripdata_2026-04.parquet')
df_zones = pd.read_csv('data/taxi_zone_lookup.csv')

print("Fetching live historical weather data from Open-Meteo API...")
url = "https://archive-api.open-meteo.com/v1/archive?latitude=40.7143&longitude=-74.006&start_date=2026-04-01&end_date=2026-04-30&hourly=precipitation"
weather_data = requests.get(url).json()

# Convert weather JSON to a Pandas DataFrame
df_weather = pd.DataFrame(weather_data['hourly'])
df_weather['time'] = pd.to_datetime(df_weather['time'])
df_weather['is_raining'] = df_weather['precipitation'] > 0

print(f"Successfully loaded {len(df_trips):,} trips, {len(df_zones)} zones, and {len(df_weather)} hours of weather data.")

Loading Parquet and CSV data...
Fetching live historical weather data from Open-Meteo API...
Successfully loaded 3,831,240 trips, 265 zones, and 720 hours of weather data.


In [6]:
# --- 2. VALIDATE & CLEAN DATA ---
initial_count = len(df_trips)

# Calculate duration in minutes
df_trips['trip_duration_minutes'] = (df_trips['tpep_dropoff_datetime'] - df_trips['tpep_pickup_datetime']).dt.total_seconds() / 60.0

# Apply rules (No teleportation, no 0 distances)
df_clean = df_trips[
    (df_trips['trip_distance'] > 0) & 
    (df_trips['trip_duration_minutes'] > 0) & 
    (df_trips['trip_duration_minutes'] < 300)
].copy()

df_clean['speed_mph'] = df_clean['trip_distance'] / (df_clean['trip_duration_minutes'] / 60.0)
df_clean = df_clean[df_clean['speed_mph'] <= 80]

print(f"Dropped {initial_count - len(df_clean):,} invalid rows.")

Dropped 144,220 invalid rows.


In [7]:
# --- 3. MODELING TEMPORAL & WEATHER WORKFLOW ---
print("Applying temporal and weather logic...")

# Map Zone Names
df_clean = df_clean.merge(df_zones[['LocationID', 'Zone']], left_on='PULocationID', right_on='LocationID', how='left')
df_clean = df_clean.rename(columns={'Zone': 'Pickup_Zone'})
df_clean = df_clean.merge(df_zones[['LocationID', 'Zone']], left_on='DOLocationID', right_on='LocationID', how='left')
df_clean = df_clean.rename(columns={'Zone': 'Dropoff_Zone'})
df_clean['Route'] = df_clean['Pickup_Zone'] + " to " + df_clean['Dropoff_Zone']

# Temporal Modeling (Rush Hour)
df_clean['pickup_hour'] = df_clean['tpep_pickup_datetime'].dt.hour
def categorize_time(hour):
    if 7 <= hour <= 9: return 'Morning Rush'
    elif 16 <= hour <= 19: return 'Evening Rush'
    else: return 'Off-Peak'
df_clean['time_of_day'] = df_clean['pickup_hour'].apply(categorize_time)

# Weather Merge (Round trip time down to nearest hour to match weather data)
df_clean['weather_join_time'] = df_clean['tpep_pickup_datetime'].dt.floor('h')
df_clean = df_clean.merge(df_weather[['time', 'is_raining']], left_on='weather_join_time', right_on='time', how='left')

df_clean[['tpep_pickup_datetime', 'time_of_day', 'is_raining', 'speed_mph']].head()

Applying temporal and weather logic...


,tpep_pickup_datetime,time_of_day,is_raining,speed_mph
0,2026-04-01 00:40:05,Off-Peak,False,13.280632
1,2026-04-01 00:09:19,Off-Peak,False,36.345205
2,2026-04-01 00:15:29,Off-Peak,False,24.512000
3,2026-04-01 00:14:20,Off-Peak,False,35.154512
4,2026-04-01 00:04:53,Off-Peak,False,11.458432


### Final Output: Proving that Rain + Rush Hour ruins NYC Traffic

In [8]:
# --- 4. ADVANCED METRICS OUTPUT ---
print("Calculating the absolute worst routes, then breaking them down by weather and time...")

# 1. First, identify the top 15 slowest routes overall (minimum 1000 trips)
route_stats = df_clean.groupby('Route').agg(
    overall_speed_mph=('speed_mph', 'mean'),
    total_trips=('speed_mph', 'count')
).reset_index()

route_stats = route_stats[route_stats['total_trips'] > 1000]
top_15_slowest = route_stats.sort_values('overall_speed_mph').head(15)
slowest_route_names = top_15_slowest['Route'].tolist()

# 2. Filter the main dataset for ONLY those 15 worst bottlenecks
df_worst = df_clean[df_clean['Route'].isin(slowest_route_names)]

# 3. Create a flat, simple table showing speed under different conditions
pivot_report = df_worst.pivot_table(
    index='Route',
    columns=['time_of_day', 'is_raining'],
    values='speed_mph',
    aggfunc='mean'
).round(2)

# Flatten the columns so it is just one big simple table
pivot_report.columns = [f"{time} (Rain: {rain})" for time, rain in pivot_report.columns]
flat_report = pivot_report.reset_index()

# Merge back the overall speed and total trips
final_table = flat_report.merge(top_15_slowest, on='Route', how='left')
final_table = final_table.sort_values('overall_speed_mph')

print("✅ Pipeline complete! Here is the clean, flat table for the DOT:")
display(final_table)

# Save to CSV
final_table.to_csv('advanced_traffic_report.csv', index=False)

Calculating the absolute worst routes, then breaking them down by weather and time...
✅ Pipeline complete! Here is the clean, flat table for the DOT:


,Route,Evening Rush (Rain: False),Evening Rush (Rain: True),Morning Rush (Rain: False),Morning Rush (Rain: True),Off-Peak (Rain: False),Off-Peak (Rain: True),overall_speed_mph,total_trips
7,Midtown East to Times Sq/Theatre District,3.86,4.00,5.15,5.03,4.79,4.81,4.496230,3458
3,Midtown Center to Times Sq/Theatre District,3.77,3.88,5.89,6.03,4.84,5.11,4.568104,4903
13,Times Sq/Theatre District to Times Sq/Theatre ...,3.98,3.68,6.21,7.10,4.91,5.06,4.672677,3402
5,Midtown East to Midtown Center,4.14,5.21,5.38,4.91,4.71,5.02,4.699963,3702
1,Midtown Center to Garment District,4.07,4.34,6.17,6.66,5.10,5.26,4.866730,2640
8,Penn Station/Madison Sq West to Garment District,4.36,3.94,5.38,5.22,4.98,4.86,4.908304,1575
14,UN/Turtle Bay South to Times Sq/Theatre District,4.32,4.84,5.98,4.90,5.13,5.49,5.019477,1100
12,Times Sq/Theatre District to Garment District,4.31,3.81,7.28,7.65,5.21,5.40,5.109556,2408
2,Midtown Center to Midtown Center,4.39,5.06,6.06,6.20,5.27,5.48,5.110237,6158
9,Penn Station/Madison Sq West to Midtown South,4.65,4.27,5.62,4.39,5.26,4.93,5.148861,2616
